# Experiment 5: Color-Textured Image Enhancement Pipeline (balanced dataset)

Eksperimen ini berfokus pada mencapai akurasi 80%+ pada dataset awan yang diratakan dengan mengombinasikan:
1. **Image Enhancement (CLAHE pada HSV + Gaussian Blur)** untuk memperjelas perbedaan kontras antara langit dan awan serta mengurangi noise.
2. **RGB/Color-based Extraction (HSV Stats & Histograms, serta NRBR Stats & Histograms)** untuk menangkap komponen warna biru langit dan putih/abu-abu awan.
3. **Texture-based Extraction (GLCM & LBP)** untuk menangkap pola tekstur mikro seperti riak altocumulus, wisps cirrus, dan kegelapan cumulonimbus.
4. Penggunaan **dataset seimbang** (`max_per_class=1250` yang di-split 80/20 menjadi 1000 train dan 250 test) untuk menghindari akurasi semu.

In [1]:
import sys
sys.path.append('..')  # supaya src/ bisa diimport dari notebooks/

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import graycomatrix, graycoprops
from src.loader import get_dataset_root, load_dataset
from src.image_processing import *

EXPERIMENT_NAME = "experiment5"


c:\Users\Mahesa\miniconda3\envs\imgproc\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


CuPy tidak aktif atau tidak berfungsi (fallback ke CPU/NumPy): Failed to find CUDA headers. Please install CUDA toolkit headers (e.g., pip install cupy-cuda12x[ctk]) or specify CUDA_PATH environment variable.


## Data Loading (Balanced to Max 1250 per Class)

In [2]:
DATASET_ROOT = get_dataset_root()
# Muat gambar warna (color=True) agar ekstraksi HSV dan NRBR bisa dilakukan
# Default max_per_class=1250 dan target_size=(256, 256) otomatis digunakan
images, labels, filenames = load_dataset(DATASET_ROOT, target_size=(256, 256), color=True)
print(f"Dataset loaded: {len(images)} gambar warna, {len(set(labels))} kelas")

Environment: Lokal
DATASET_ROOT dari .env: D:\INFORMATICS\SEMESTER 4\IMAGE PROCESSING\Praktikum\Project\GCD


Loading altocumulus:   0%|          | 0/1250 [00:00<?, ?it/s]

Loading altocumulus:   5%|▌         | 63/1250 [00:00<00:01, 615.47it/s]

Loading altocumulus:  10%|█         | 126/1250 [00:00<00:01, 623.72it/s]

Loading altocumulus:  15%|█▌        | 189/1250 [00:00<00:01, 609.79it/s]

...

Loading stratocumulus:  83%|████████▎ | 1038/1250 [00:01<00:00, 592.43it/s]

Loading stratocumulus:  88%|████████▊ | 1098/1250 [00:01<00:00, 571.65it/s]

Loading stratocumulus:  92%|█████████▏| 1156/1250 [00:02<00:00, 542.53it/s]

Loading stratocumulus:  97%|█████████▋| 1211/1250 [00:02<00:00, 536.07it/s]

Dataset loaded: 8455 gambar, 7 kelas, ukuran=(256, 256)
Dataset loaded: 8455 gambar warna, 7 kelas


## Preprocessing Pipeline (CLAHE HSV + Gaussian Blur)

In [3]:
# ================================================================
# EDIT BAGIAN INI untuk mengubah pipeline preprocessing
# Tambah, hapus, atau ganti fungsi sesuai experiment
# ================================================================

PIPELINE = [
    lambda img: clahe_hsv(img, tile_size=8, clip_limit=2.0),
    lambda img: gaussian_filter(img, kernel_size=5, sigma=1.0),
]

# ================================================================
# Jangan edit di bawah ini
# ================================================================

def apply_pipeline(image: np.ndarray, pipeline: list) -> np.ndarray:
    for fn in pipeline:
        image = fn(image)
    return image


In [4]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

def process_one(img):
    return apply_pipeline(img, PIPELINE)

print("Menjalankan Preprocessing Pipeline...")
if USING_GPU:
    print("Menggunakan ThreadPoolExecutor (GPU aktif)...")
    with ThreadPoolExecutor(max_workers=4) as executor:
        images_preprocessed_color = list(tqdm(
            executor.map(process_one, images),
            total=len(images),
            desc="Preprocessing"
        ))
else:
    print("Menggunakan Pemrosesan Sekuensial CPU...")
    images_preprocessed_color = [process_one(img) for img in tqdm(images, desc="Preprocessing")]
    
images_preprocessed_color = np.array(images_preprocessed_color)

Menjalankan Preprocessing Pipeline...


Menggunakan Pemrosesan Sekuensial CPU...


Preprocessing:   0%|          | 0/8455 [00:00<?, ?it/s]

Preprocessing:   0%|          | 2/8455 [00:00<08:59, 15.67it/s]

Preprocessing:   0%|          | 5/8455 [00:00<06:45, 20.83it/s]

...

Preprocessing: 100%|█████████▉| 8452/8455 [11:16<00:00,  8.50it/s]

Preprocessing: 100%|█████████▉| 8453/8455 [11:16<00:00,  8.18it/s]

Preprocessing: 100%|█████████▉| 8454/8455 [11:16<00:00,  8.01it/s]

Preprocessing: 100%|██████████| 8455/8455 [11:16<00:00,  7.88it/s]

Preprocessing: 100%|██████████| 8455/8455 [11:16<00:00, 12.50it/s]

## Visualisasi Preprocessing Bertahap

Melihat efek dari setiap tahapan preprocessing secara bertahap.

In [ ]:
# Visualisasi Preprocessing Bertahap secara otomatis untuk semua tahapan
visualize_pipeline_steps(images, labels, PIPELINE)


In [5]:
# Konversi hasil preprocessing ke grayscale khusus untuk ekstraksi fitur GLCM & LBP
print("Mengonversi gambar preprocessed ke grayscale...")
images_preprocessed = np.array([
    cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in tqdm(images_preprocessed_color, desc="Converting to grayscale")
])

Mengonversi gambar preprocessed ke grayscale...


Converting to grayscale:   0%|          | 0/8455 [00:00<?, ?it/s]

Converting to grayscale:   1%|▏         | 123/8455 [00:00<00:06, 1227.83it/s]

Converting to grayscale:   3%|▎         | 263/8455 [00:00<00:06, 1321.23it/s]

Converting to grayscale:   5%|▍         | 396/8455 [00:00<00:06, 1225.49it/s]

...

Converting to grayscale:  62%|██████▏   | 5202/8455 [00:01<00:00, 7708.77it/s]

Converting to grayscale:  72%|███████▏  | 6078/8455 [00:01<00:00, 8020.22it/s]

Converting to grayscale:  83%|████████▎ | 6996/8455 [00:01<00:00, 8355.17it/s]

Converting to grayscale:  94%|█████████▎| 7907/8455 [00:01<00:00, 8532.69it/s]

Converting to grayscale: 100%|██████████| 8455/8455 [00:01<00:00, 6081.26it/s]

## Feature Extraction - Color & Texture

Kita mengekstrak:
- **HSV Color Stats & Histograms**
- **NRBR Color Stats & Histograms**
- **LBP Texture Histograms**
- **GLCM Texture Features**

In [6]:
# Ekstrasi Fitur Warna HSV
print("Mengekstrak fitur HSV...")
h_hists, s_hists, v_hists, \
h_means, h_stds, h_skews, h_kurts, \
s_means, s_stds, s_skews, s_kurts, \
v_means, v_stds, v_skews, v_kurts = extract_hsv_features(images_preprocessed_color)

# Ekstraksi Fitur Warna NRBR (Normalized Red-Blue Ratio)
print("Mengekstrak fitur NRBR...")
nrbr_hists, nrbr_means, nrbr_stds, nrbr_skews, nrbr_kurts = extract_nrbr_features(images_preprocessed_color)

# Ekstraksi Fitur Tekstur LBP (Local Binary Pattern)
print("Mengekstrak fitur LBP...")
lbp_hists = extract_lbp_features(images_preprocessed, bins=8)

Mengekstrak fitur HSV...


Mengekstrak fitur NRBR...


Mengekstrak fitur LBP...


In [7]:
# Ekstraksi Fitur Tekstur GLCM
def glcm(image, angle):
    if angle == 0:
        angles = [0]
    elif angle == 45:
        angles = [np.pi / 4]
    elif angle == 90:
        angles = [np.pi / 2]
    elif angle == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Sudut tidak valid. Gunakan 0, 45, 90, atau 135.")
    return graycomatrix(image, [1], angles, 256, symmetric=True, normed=True)

def correlation(matriks):
    return graycoprops(matriks, 'correlation')[0, 0]
def dissimilarity(matriks):
    return graycoprops(matriks, 'dissimilarity')[0, 0]
def homogenity(matriks):
    return graycoprops(matriks, 'homogeneity')[0, 0]
def contrast(matriks):
    return graycoprops(matriks, 'contrast')[0, 0]
def ASM(matriks):
    return graycoprops(matriks, 'ASM')[0, 0]
def energy(matriks):
    return graycoprops(matriks, 'energy')[0, 0]
def entropyGlcm(matriks):
    return entropy(matriks.ravel())

Kontras0, Kontras45, Kontras90, Kontras135 = [], [], [], []
dissimilarity0, dissimilarity45, dissimilarity90, dissimilarity135 = [], [], [], []
homogenity0, homogenity45, homogenity90, homogenity135 = [], [], [], []
entropy0, entropy45, entropy90, entropy135 = [], [], [], []
ASM0, ASM45, ASM90, ASM135 = [], [], [], []
energy0, energy45, energy90, energy135 = [], [], [], []
correlation0, correlation45, correlation90, correlation135 = [], [], [], []

for i in tqdm(range(len(images_preprocessed)), desc="Extracting GLCM features"):
    g0 = glcm(images_preprocessed[i], 0)
    g45 = glcm(images_preprocessed[i], 45)
    g90 = glcm(images_preprocessed[i], 90)
    g135 = glcm(images_preprocessed[i], 135)
    
    Kontras0.append(contrast(g0))
    Kontras45.append(contrast(g45))
    Kontras90.append(contrast(g90))
    Kontras135.append(contrast(g135))
    
    dissimilarity0.append(dissimilarity(g0))
    dissimilarity45.append(dissimilarity(g45))
    dissimilarity90.append(dissimilarity(g90))
    dissimilarity135.append(dissimilarity(g135))
    
    homogenity0.append(homogenity(g0))
    homogenity45.append(homogenity(g45))
    homogenity90.append(homogenity(g90))
    homogenity135.append(homogenity(g135))
    
    entropy0.append(entropyGlcm(g0))
    entropy45.append(entropyGlcm(g45))
    entropy90.append(entropyGlcm(g90))
    entropy135.append(entropyGlcm(g135))
    
    ASM0.append(ASM(g0))
    ASM45.append(ASM(g45))
    ASM90.append(ASM(g90))
    ASM135.append(ASM(g135))
    
    energy0.append(energy(g0))
    energy45.append(energy(g45))
    energy90.append(energy(g90))
    energy135.append(energy(g135))
    
    correlation0.append(correlation(g0))
    correlation45.append(correlation(g45))
    correlation90.append(correlation(g90))
    correlation135.append(correlation(g135))

Extracting GLCM features:   0%|          | 0/8455 [00:00<?, ?it/s]

Extracting GLCM features:   0%|          | 4/8455 [00:00<03:42, 38.02it/s]

Extracting GLCM features:   0%|          | 9/8455 [00:00<03:39, 38.51it/s]

Extracting GLCM features:   0%|          | 14/8455 [00:00<03:15, 43.07it/s]

Extracting GLCM features:   0%|          | 20/8455 [00:00<02:57, 47.47it/s]

...

Extracting GLCM features: 100%|█████████▉| 8430/8455 [02:28<00:00, 58.88it/s]

Extracting GLCM features: 100%|█████████▉| 8437/8455 [02:28<00:00, 61.51it/s]

Extracting GLCM features: 100%|█████████▉| 8444/8455 [02:29<00:00, 60.79it/s]

Extracting GLCM features: 100%|█████████▉| 8451/8455 [02:29<00:00, 60.60it/s]

Extracting GLCM features: 100%|██████████| 8455/8455 [02:29<00:00, 56.66it/s]

In [8]:
# Gabungkan semua fitur ke dalam satu DataFrame
dataTable = {
    'Filename': filenames,
    'Label': labels,
    # GLCM
    'Contrast0': Kontras0, 'Contrast45': Kontras45, 'Contrast90': Kontras90, 'Contrast135': Kontras135,
    'Homogeneity0': homogenity0, 'Homogeneity45': homogenity45, 'Homogeneity90': homogenity90, 'Homogeneity135': homogenity135,
    'Dissimilarity0': dissimilarity0, 'Dissimilarity45': dissimilarity45, 'Dissimilarity90': dissimilarity90, 'Dissimilarity135': dissimilarity135,
    'Entropy0': entropy0, 'Entropy45': entropy45, 'Entropy90': entropy90, 'Entropy135': entropy135,
    'ASM0': ASM0, 'ASM45': ASM45, 'ASM90': ASM90, 'ASM135': ASM135,
    'Energy0': energy0, 'Energy45': energy45, 'Energy90': energy90, 'Energy135': energy135,
    'Correlation0': correlation0, 'Correlation45': correlation45, 'Correlation90': correlation90, 'Correlation135': correlation135,
    # HSV Stats
    'HSV_H_Mean': h_means, 'HSV_H_Std': h_stds, 'HSV_H_Skew': h_skews, 'HSV_H_Kurt': h_kurts,
    'HSV_S_Mean': s_means, 'HSV_S_Std': s_stds, 'HSV_S_Skew': s_skews, 'HSV_S_Kurt': s_kurts,
    'HSV_V_Mean': v_means, 'HSV_V_Std': v_stds, 'HSV_V_Skew': v_skews, 'HSV_V_Kurt': v_kurts,
    # NRBR Stats
    'NRBR_Mean': nrbr_means, 'NRBR_Std': nrbr_stds, 'NRBR_Skew': nrbr_skews, 'NRBR_Kurt': nrbr_kurts,
}

# Tambahkan HSV Histograms
for b in range(16):
    dataTable[f'HSV_H_Hist_Bin_{b}'] = h_hists[:, b]
for b in range(8):
    dataTable[f'HSV_S_Hist_Bin_{b}'] = s_hists[:, b]
for b in range(8):
    dataTable[f'HSV_V_Hist_Bin_{b}'] = v_hists[:, b]

# Tambahkan NRBR Histograms
for b in range(16):
    dataTable[f'NRBR_Hist_Bin_{b}'] = nrbr_hists[:, b]

# Tambahkan LBP Histograms
for b in range(8):
    dataTable[f'LBP_Hist_Bin_{b}'] = lbp_hists[:, b]

df = pd.DataFrame(dataTable)
csv_path = f'hasil_ekstraksi_{EXPERIMENT_NAME}.csv'
df.to_csv(csv_path, index=False)
print(f'Fitur gabungan berhasil disimpan ke {csv_path}')
df.head()

Fitur gabungan berhasil disimpan ke hasil_ekstraksi_experiment11.csv


,Filename,Label,Contrast0,Contrast45,Contrast90,Contrast135,Homogeneity0,Homogeneity45,Homogeneity90,Homogeneity135,...,NRBR_Hist_Bin_14,NRBR_Hist_Bin_15,LBP_Hist_Bin_0,LBP_Hist_Bin_1,LBP_Hist_Bin_2,LBP_Hist_Bin_3,LBP_Hist_Bin_4,LBP_Hist_Bin_5,LBP_Hist_Bin_6,LBP_Hist_Bin_7
0,2_altocumulus_001371.jpg,altocumulus,45.162577,75.582253,59.245910,121.971334,0.447410,0.359880,0.417410,0.355261,...,0.0,0.0,0.004988,0.002035,0.000191,0.003497,0.002592,0.000248,0.003528,0.014171
1,2_altocumulus_000676.jpg,altocumulus,49.989017,116.052349,76.616268,129.307805,0.196684,0.133417,0.159579,0.128522,...,0.0,0.0,0.009366,0.003005,0.000189,0.003342,0.003275,0.000199,0.003188,0.008687
2,2_altocumulus_001452.jpg,altocumulus,48.850169,109.175840,63.478248,111.653641,0.216695,0.145136,0.196193,0.160454,...,0.0,0.0,0.008274,0.003619,0.000090,0.003450,0.003171,0.000108,0.003575,0.008963
3,2_altocumulus_000336.jpg,altocumulus,72.228294,151.205506,106.456679,193.714295,0.365773,0.281365,0.305556,0.252029,...,0.0,0.0,0.007523,0.002544,0.000189,0.003487,0.002777,0.000214,0.002666,0.011851
4,2_altocumulus_000201.jpg,altocumulus,32.023621,99.316417,86.434528,132.880415,0.303837,0.210173,0.234145,0.192863,...,0.0,0.0,0.009930,0.001874,0.000150,0.002567,0.002887,0.000167,0.003197,0.010479


## Train-Test Split & Normalization

In [9]:
# Pisahkan fitur dan label
X = df.drop(columns=['Filename', 'Label']).values
y = df['Label'].values

# Split 80/20 secara acak dan terstratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Z-score Normalization (dengan epsilon 1e-8 untuk menghindari pembagian dengan nol)
mean_train = X_train.mean(axis=0)
std_train = X_train.std(axis=0)
X_train_norm = (X_train - mean_train) / (std_train + 1e-8)
X_test_norm = (X_test - mean_train) / (std_train + 1e-8)
print(f"Train shape: {X_train_norm.shape}")
print(f"Test shape: {X_test_norm.shape}")

Train shape: (6764, 100)
Test shape: (1691, 100)


## Classifiers Training & Evaluation

In [10]:
def evaluate_model(clf, X_tr, y_tr, X_ts, y_ts, name):
    clf.fit(X_tr, y_tr)
    y_pred_tr = clf.predict(X_tr)
    y_pred_ts = clf.predict(X_ts)
    
    print(f"\n====== {name} - Training Set ======")
    print(classification_report(y_tr, y_pred_tr, zero_division=0))
    print(f"Accuracy: {accuracy_score(y_tr, y_pred_tr):.4f}")
    
    print(f"\n====== {name} - Testing Set ======")
    print(classification_report(y_ts, y_pred_ts, zero_division=0))
    print(confusion_matrix(y_ts, y_pred_ts))
    print(f"Accuracy: {accuracy_score(y_ts, y_pred_ts):.4f}")
    return y_pred_ts

# Definisikan model dengan class_weight='balanced' untuk penanganan optimal jika ada minor imbalans
rf = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
svm = SVC(kernel='rbf', C=2.0, random_state=42, class_weight='balanced')
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')

y_pred_rf_test = evaluate_model(rf, X_train_norm, y_train, X_test_norm, y_test, "Random Forest")
y_pred_svm_test = evaluate_model(svm, X_train_norm, y_train, X_test_norm, y_test, "SVM")
y_pred_knn_test = evaluate_model(knn, X_train_norm, y_train, X_test_norm, y_test, "KNN")


====== Random Forest - Training Set ======
               precision    recall  f1-score   support

  altocumulus       1.00      1.00      1.00      1000
       cirrus       1.00      1.00      1.00      1000
     clearsky       1.00      1.00      1.00      1000
 cumulonimbus       1.00      1.00      1.00      1000
      cumulus       1.00      1.00      1.00      1000
        mixed       1.00      1.00      1.00       764
stratocumulus       1.00      1.00      1.00      1000

     accuracy                           1.00      6764
    macro avg       1.00      1.00      1.00      6764
 weighted avg       1.00      1.00      1.00      6764

Accuracy: 1.0000

====== Random Forest - Testing Set ======
               precision    recall  f1-score   support

  altocumulus       0.84      0.87      0.86       250
       cirrus       0.82      0.77      0.80       250
     clearsky       0.96      0.96      0.96       250
 cumulonimbus       0.75      0.79      0.77       250
      cumulu


====== SVM - Training Set ======
               precision    recall  f1-score   support

  altocumulus       0.89      0.87      0.88      1000
       cirrus       0.86      0.87      0.86      1000
     clearsky       0.98      0.99      0.98      1000
 cumulonimbus       0.86      0.85      0.86      1000
      cumulus       0.92      0.91      0.91      1000
        mixed       0.79      0.79      0.79       764
stratocumulus       0.83      0.85      0.84      1000

     accuracy                           0.88      6764
    macro avg       0.87      0.87      0.87      6764
 weighted avg       0.88      0.88      0.88      6764

Accuracy: 0.8771

====== SVM - Testing Set ======
               precision    recall  f1-score   support

  altocumulus       0.86      0.88      0.87       250
       cirrus       0.83      0.80      0.81       250
     clearsky       0.96      0.97      0.96       250
 cumulonimbus       0.76      0.79      0.78       250
      cumulus       0.89      0.


====== KNN - Training Set ======
               precision    recall  f1-score   support

  altocumulus       1.00      1.00      1.00      1000
       cirrus       1.00      1.00      1.00      1000
     clearsky       1.00      1.00      1.00      1000
 cumulonimbus       1.00      1.00      1.00      1000
      cumulus       1.00      1.00      1.00      1000
        mixed       1.00      1.00      1.00       764
stratocumulus       1.00      1.00      1.00      1000

     accuracy                           1.00      6764
    macro avg       1.00      1.00      1.00      6764
 weighted avg       1.00      1.00      1.00      6764

Accuracy: 1.0000

====== KNN - Testing Set ======
               precision    recall  f1-score   support

  altocumulus       0.79      0.83      0.81       250
       cirrus       0.78      0.69      0.73       250
     clearsky       0.95      0.96      0.95       250
 cumulonimbus       0.66      0.78      0.72       250
      cumulus       0.82      0.

## Save Experiment Metrics & Auto-update README

In [11]:
from pathlib import Path
METRICS_PATH = Path('../results/metrics.csv')
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

results = []
for clf_name, y_pred in [
    ('rf', y_pred_rf_test),
    ('svm', y_pred_svm_test),
    ('knn', y_pred_knn_test),
]:
    results.append({
        'experiment_name': EXPERIMENT_NAME,
        'classifier': clf_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    })

new_rows = pd.DataFrame(results)

if METRICS_PATH.exists() and METRICS_PATH.stat().st_size > 0:
    all_metrics = pd.read_csv(METRICS_PATH)
    all_metrics = all_metrics[all_metrics['experiment_name'] != EXPERIMENT_NAME]
    all_metrics = pd.concat([all_metrics, new_rows], ignore_index=True)
else:
    all_metrics = new_rows

all_metrics.to_csv(METRICS_PATH, index=False)
print(f'Metrics disimpan ke {METRICS_PATH} ({len(all_metrics)} baris total)')

# Jalankan script generate_metrics_table.py untuk memperbarui tabel hasil di README.md secara otomatis
import subprocess
try:
    subprocess.run(["python", "../src/generate_metrics_table.py"], check=True)
    print("README.md berhasil diperbarui dengan tabel hasil terbaru!")
except Exception as e:
    print(f"Peringatan: Gagal meng-update README.md: {e}")

all_metrics

Metrics disimpan ke ..\results\metrics.csv (33 baris total)


README.md berhasil diperbarui dengan tabel hasil terbaru!


,experiment_name,classifier,accuracy,precision,recall,f1
0,experiment9,rf,0.575789,0.564385,0.575789,0.562935
1,experiment9,svm,0.563158,0.548167,0.563158,0.531862
2,experiment9,knn,0.522632,0.506224,0.522632,0.506076
3,experiment1,rf,0.532105,0.519583,0.532105,0.516108
4,experiment1,svm,0.533158,0.532211,0.533158,0.493381
5,experiment1,knn,0.483684,0.464744,0.483684,0.464786
6,experiment2,rf,0.553421,0.537102,0.553421,0.540455
7,experiment2,svm,0.540789,0.528985,0.540789,0.512945
8,experiment2,knn,0.516316,0.503454,0.516316,0.505084
9,experiment3,rf,0.561842,0.550465,0.561842,0.550986


## Analisis Eksperimen 11

Silakan berikan ulasan analisis Anda di sini.